In [ ]:
import pandas as pd
import numpy as np

# Loading data
file_path = '../data/investigator_nacc72.csv'
df_raw = pd.read_csv(file_path, low_memory=False)

In [ ]:
# MMSE bereinigen (nur Werte 0-30 sind gültig, Rest ist NaN)
df_raw['MMSE_v'] = df_raw['NACCMMSE'].where((df_raw['NACCMMSE'] >= 0) & (df_raw['NACCMMSE'] <= 30))

# Baseline bestimmen (Erster Visit)
df_bl = df_raw[df_raw['NACCVNUM'] == 1][['NACCID', 'SEX', 'NACCAGE', 'NACCUDSD', 'MMSE_v']].copy()
df_bl = df_bl.rename(columns={'NACCUDSD': 'UDSD_bl', 'NACCAGE': 'Age', 'MMSE_v': 'MMSE_bl'})

# Follow-up bestimmen (Letzter Visit)
df_fu = df_raw.sort_values(['NACCID', 'NACCVNUM']).groupby('NACCID').tail(1)[['NACCID', 'NACCUDSD', 'NACCDAYS', 'MMSE_v']].copy()
df_fu = df_fu.rename(columns={'NACCUDSD': 'UDSD_fu', 'MMSE_v': 'MMSE_fu'})

# Join zu Transitionen
df = pd.merge(df_bl, df_fu, on='NACCID')
df = df[df['NACCDAYS'] > 0].copy()

# Engineering
df['FollowUp'] = df['NACCDAYS'] / 365.25
df['Gender'] = df['SEX'].map({1: 'M', 2: 'F'})
df['ParticipantID'] = df['NACCID']

mapping = {1: 'NL', 2: 'I.,nMCI', 3: 'MCI', 4: 'AD'}
df['temp_bl'] = df['UDSD_bl'].map(mapping)
df['temp_fu'] = df['UDSD_fu'].map(mapping)

df['Group'] = df['temp_bl'].astype(str) + ' to ' + df['temp_fu'].astype(str)

mci_prog = (df['UDSD_bl'] == 3) & (df['UDSD_fu'] == 3) & (df['MMSE_fu'] < df['MMSE_bl'])
df.loc[mci_prog, 'Group'] = 'MCI Progression'

mci_rev = (df['UDSD_bl'] == 3) & (df['UDSD_fu'] == 3) & (df['MMSE_fu'] > df['MMSE_bl'])
df.loc[mci_rev, 'Group'] = 'MCI Reversal'

In [ ]:
# List filter
forward_groups = ['NL to I.,nMCI', 'I.,nMCI to MCI', 'MCI to AD', 'NL to MCI', 'I.,nMCI to AD', 'NL to AD', 'MCI Progression']
backward_groups = ['AD to MCI', 'MCI to I.,nMCI', 'I.,nMCI to NL', 'MCI to NL', 'AD to I.,nMCI', 'AD to NL', 'MCI Reversal']

# Berechnung

# Hilfsfunktion für die Zeilenberechnung
forward_results = []
for g in forward_groups:
    subset = df[df['Group'] == g]
    if subset.empty: continue
    forward_results.append({
        'Group': g,
        'N unique participants': subset['ParticipantID'].nunique(),
        'Female / Male': f"{len(subset[subset['Gender'] == 'F'])} / {len(subset[subset['Gender'] == 'M'])}",
        'Follow-up years (mean ± SE)': f"{subset['FollowUp'].mean():.1f} ± {subset['FollowUp'].sem():.1f}",
        'Age at baseline (mean ± SE)': f"{subset['Age'].mean():.1f} ± {subset['Age'].sem():.1f}"
    })

# TOTAL für Forward
forward_all = df[df['Group'].isin(forward_groups)]
if not forward_all.empty:
    forward_results.append({
        'Group': 'TOTAL',
        'N unique participants': forward_all['ParticipantID'].nunique(),
        'Female / Male': f"{len(forward_all[forward_all['Gender'] == 'F'])} / {len(forward_all[forward_all['Gender'] == 'M'])}",
        'Follow-up years (mean ± SE)': f"{forward_all['FollowUp'].mean():.1f} ± {forward_all['FollowUp'].sem():.1f}",
        'Age at baseline (mean ± SE)': f"{forward_all['Age'].mean():.1f} ± {forward_all['Age'].sem():.1f}"
    })

backward_results = []
for g in backward_groups:
    subset = df[df['Group'] == g]
    if subset.empty: continue
    backward_results.append({
        'Group': g,
        'N unique participants': subset['ParticipantID'].nunique(),
        'Female / Male': f"{len(subset[subset['Gender'] == 'F'])} / {len(subset[subset['Gender'] == 'M'])}",
        'Follow-up years (mean ± SE)': f"{subset['FollowUp'].mean():.1f} ± {subset['FollowUp'].sem():.1f}",
        'Age at baseline (mean ± SE)': f"{subset['Age'].mean():.1f} ± {subset['Age'].sem():.1f}"
    })

# TOTAL für Backward
backward_all = df[df['Group'].isin(backward_groups)]
if not backward_all.empty:
    backward_results.append({
        'Group': 'TOTAL',
        'N unique participants': backward_all['ParticipantID'].nunique(),
        'Female / Male': f"{len(backward_all[backward_all['Gender'] == 'F'])} / {len(backward_all[backward_all['Gender'] == 'M'])}",
        'Follow-up years (mean ± SE)': f"{backward_all['FollowUp'].mean():.1f} ± {backward_all['FollowUp'].sem():.1f}",
        'Age at baseline (mean ± SE)': f"{backward_all['Age'].mean():.1f} ± {backward_all['Age'].sem():.1f}"
    })

In [ ]:
# Stable groups (Category hasn't changed)
stable_groups = ['NL to NL', 'I.,nMCI to I.,nMCI', 'MCI to MCI', 'AD to AD']

stable_results = []
for g in stable_groups:
    subset = df[df['Group'] == g]
    if subset.empty: continue
    stable_results.append({
        'Group': g,
        'N unique participants': subset['ParticipantID'].nunique(),
        'Female / Male': f"{len(subset[subset['Gender'] == 'F'])} / {len(subset[subset['Gender'] == 'M'])}",
        'Follow-up years (mean ± SE)': f"{subset['FollowUp'].mean():.1f} ± {subset['FollowUp'].sem():.1f}",
        'Age at baseline (mean ± SE)': f"{subset['Age'].mean():.1f} ± {subset['Age'].sem():.1f}"
    })

# Add TOTAL for Stable if any
stable_all = df[df['Group'].isin(stable_groups)]
if not stable_all.empty:
    stable_results.append({
        'Group': 'TOTAL STABLE',
        'N unique participants': stable_all['ParticipantID'].nunique(),
        'Female / Male': f"{len(stable_all[stable_all['Gender'] == 'F'])} / {len(stable_all[stable_all['Gender'] == 'M'])}",
        'Follow-up years (mean ± SE)': f"{stable_all['FollowUp'].mean():.1f} ± {stable_all['FollowUp'].sem():.1f}",
        'Age at baseline (mean ± SE)': f"{stable_all['Age'].mean():.1f} ± {stable_all['Age'].sem():.1f}"
    })

In [13]:
# Display
print("Forward Groups Analysis / Cognitive regression")
display(pd.DataFrame(forward_results))

print("\nBackward Groups Analysis / Cognitive progression")
display(pd.DataFrame(backward_results))

print("Stable Groups Analysis")
display(pd.DataFrame(stable_results))

Forward Groups Analysis / Cognitive regression


,Group,N unique participants,Female / Male,Follow-up years (mean ± SE),Age at baseline (mean ± SE)
0,"NL to I.,nMCI",629,395 / 234,6.8 ± 0.2,71.2 ± 0.4
1,"I.,nMCI to MCI",401,219 / 182,5.5 ± 0.2,72.6 ± 0.5
2,MCI to AD,3470,1641 / 1829,4.9 ± 0.1,74.0 ± 0.2
3,NL to MCI,2113,1307 / 806,6.9 ± 0.1,74.2 ± 0.2
4,"I.,nMCI to AD",294,163 / 131,7.3 ± 0.2,73.8 ± 0.6
5,NL to AD,1289,813 / 476,9.3 ± 0.1,76.6 ± 0.3
6,MCI Progression,607,315 / 292,2.8 ± 0.1,74.2 ± 0.4
7,TOTAL,8803,4853 / 3950,6.1 ± 0.0,74.2 ± 0.1



Backward Groups Analysis / Cognitive progression


,Group,N unique participants,Female / Male,Follow-up years (mean ± SE),Age at baseline (mean ± SE)
0,AD to MCI,213,94 / 119,3.8 ± 0.2,69.6 ± 0.7
1,"MCI to I.,nMCI",379,189 / 190,4.5 ± 0.2,70.0 ± 0.4
2,"I.,nMCI to NL",515,304 / 211,5.4 ± 0.2,67.3 ± 0.4
3,MCI to NL,1020,568 / 452,5.1 ± 0.1,69.0 ± 0.3
4,"AD to I.,nMCI",52,17 / 35,5.7 ± 0.6,65.2 ± 1.5
5,AD to NL,55,23 / 32,5.5 ± 0.7,67.1 ± 1.2
6,MCI Reversal,326,160 / 166,2.6 ± 0.1,73.3 ± 0.5
7,TOTAL,2560,1355 / 1205,4.7 ± 0.1,69.3 ± 0.2


Stable Groups Analysis


,Group,N unique participants,Female / Male,Follow-up years (mean ± SE),Age at baseline (mean ± SE)
0,NL to NL,13141,8694 / 4447,5.7 ± 0.0,68.7 ± 0.1
1,"I.,nMCI to I.,nMCI",437,264 / 173,4.2 ± 0.2,68.7 ± 0.5
2,MCI to MCI,2703,1324 / 1379,3.7 ± 0.1,71.7 ± 0.2
3,AD to AD,10291,5202 / 5089,3.3 ± 0.0,71.9 ± 0.1
4,TOTAL STABLE,26572,15484 / 11088,4.5 ± 0.0,70.3 ± 0.1
